# Text Fundamentals

> 📘 **Python Mastery** · Module 16 — NLP · Lesson 1/5

Every chat message, review, and support ticket is just a sequence of characters — and before any model can learn from language, we must treat those characters as data: counting them, cleaning them, and measuring them.

## 🎯 Learning Objectives

- Explain why natural language needs special handling before machines can learn from it
- Manipulate strings with the methods real text work relies on: `lower`, `strip`, `split`, `join`, `replace`, `startswith`, slicing
- Handle Unicode confidently: `ord`/`chr`, characters vs bytes, emoji, and normalization gotchas
- Read text from files with the right encoding and assemble documents into a **corpus**
- Compute corpus statistics — total words, vocabulary size, frequency distribution, lexical richness
- Split sentences with regex, see exactly why naive patterns fail, and repair them with lookbehind

## 1. Text Is Data

To Python, `"Loved it!!!"` is not an opinion yet — it is 11 characters. NLP starts by converting messy human text into clean, countable pieces that downstream math can consume. Raw text is like a bag of unsorted groceries: before cooking, you wash, peel, and chop.

Real text arrives messy in predictable ways:

| Messiness | Example | Why it hurts |
|---|---|---|
| Inconsistent casing | `GREAT`, `great`, `Great` | machine sees 3 different words |
| Stray whitespace | `"  hi   there "` | fake differences between texts |
| Punctuation glued to words | `"great!"` ≠ `"great"` | vocabulary explodes |
| Non-ASCII symbols | `é`, `🙂`, `🇧🇩` | behave surprisingly in bytes and slicing |

In [ ]:
raw_review = "  LOVED it!!! Best purchase ever 😍  "
clean_ish = "loved it best purchase ever"

print(repr(raw_review))          # repr() reveals hidden spaces at both ends
words = raw_review.lower().strip().split()
print(words)                     # already much closer to the clean version
print(len(raw_review), "characters including spaces")

## 2. Strings Recap — For Text Work

You already know string methods from earlier modules. Here we sharpen exactly the subset that NLP code uses dozens of times per pipeline: case folding, trimming, splitting/gluing, replacing, prefix tests, and slicing.

**Syntax:**

```python
text.lower()               # fold case ("LOVED" -> "loved")
text.strip()               # trim whitespace from both ends
text.split()               # split on runs of whitespace -> list of words
"-".join(words)            # glue a list back into one string
text.replace(a, b)         # swap every occurrence of a for b
text.startswith(p)         # True if text begins with p
text[0:10]                 # slice: first ten characters
len(text)                  # number of characters
```

In [ ]:
ticket = "  ORDER #4521 has not arrived. Please HELP!  "

print(ticket.lower())               # lowercase -> safe matching later
print(ticket.strip())               # trim the edge whitespace
print(ticket.startswith("ORDER"))   # routing rule: order tickets start with ORDER
print("#" in ticket)                # membership test finds the order marker

words = ticket.strip().split()
print(words[:4])                    # slicing: first four words
print("-".join(["urgent", "order", "4521"]))      # building ids / keys
print(ticket.strip().replace("#", "No."))         # normalise the symbol

### 2.1 Strings Are Immutable

String methods never change the original — they return a *new* string. That is why we chain calls: each link in the chain hands its result to the next method.

In [ ]:
name = "Sarah"
loud_name = name.upper()

print(name)       # unchanged - the original string is untouched
print(loud_name)  # upper() returned a brand-new string

# Chaining works because every call returns the new string:
print("  Hello World ".strip().lower().replace("world", "Dhaka"))

## 3. Unicode & UTF-8

Computers store bytes; humans write letters. A Python 3 `str` is a sequence of Unicode **code points** (numbers), and UTF-8 is the standard recipe for turning those numbers into bytes on disk or over the network. Two functions bridge the two worlds.

**Syntax:**

```python
ord(c)                     # character -> code point number
chr(n)                     # code point number -> character
s.encode("utf-8")          # str -> bytes
b.decode("utf-8")          # bytes -> str
len(s)                     # counts CODE POINTS, not bytes
```

In [ ]:
print(ord("A"), ord("a"), ord("🙂"))   # code points: 65, 97, and a big number
print(chr(2535))                       # Bengali digit one - any number works

smiley = "🙂"
print(smiley, "->", len(smiley), "character")
print(smiley.encode("utf-8"), "->", len(smiley.encode("utf-8")), "bytes")

flag = "🇧🇩"                            # Bangladesh flag emoji
print(flag, "has length:", len(flag))  # surprise: TWO code points!

Why does `len` matter? `len(s)` counts code points, while UTF-8 spends 1 byte on plain ASCII but up to 4 bytes on emoji. The flag `🇧🇩` is not one symbol internally — it is two *regional indicator* code points (`ord("🇧")` style pairs) that fonts merge visually. Slicing cuts code points, so it can tear such sequences apart.

In [ ]:
greeting = "hello 🙂"

print(len(greeting), "characters,", len(greeting.encode("utf-8")), "bytes")

print(greeting[:7])        # character slicing keeps the emoji intact
cut = greeting.encode("utf-8")[:7]     # byte slicing lands mid-emoji...
print(cut)
print(cut.decode("utf-8", errors="replace"))   # ...so decoding produces garbage

### 3.1 Normalization Gotchas

The same visible letter can have more than one internal spelling. `é` can be **composed** (one code point `U+00E9`) or **decomposed** (`e` plus combining accent `U+0301`). To your eyes they match; to `==` they do not. `unicodedata.normalize` fixes this.

In [ ]:
import unicodedata

e1 = "café"          # composed: é is ONE code point (U+00E9)
e2 = "cafe\u0301"    # decomposed: e + COMBINING ACUTE ACCENT

print(e1 == e2, "| lengths:", len(e1), "vs", len(e2))
print([hex(ord(ch)) for ch in e1])
print([hex(ord(ch)) for ch in e2])

nfc = unicodedata.normalize("NFC", e2)   # pull accents onto the base letter
print("after NFC:", nfc == e1, "length:", len(nfc))

In [ ]:
import unicodedata

e1 = "café"
nfd = unicodedata.normalize("NFD", e1)   # push apart: base letter + accent
print(nfd, "has length:", len(nfd))

# Rule of thumb: normalize EVERYTHING once, early, always the same way.
def canon(text):
    return unicodedata.normalize("NFC", text)

search_log = ["café", "cafe\u0301", "Café"]
unique = {canon(s).lower() for s in search_log}
print(sorted(unique), "<- three spellings collapsed to two")

> 🔍 **Under the Hood:** CPython does not store every string as 4 bytes per character. Since PEP 393 it picks the narrowest internal layout that fits the content: 1 byte per char for pure ASCII ("kind 1"), 2 bytes when any char fits in two bytes, 4 bytes otherwise (emoji!). That is why `len()` is instant — the length sits in the string header — while memory size jumps once you add a single emoji. Watch it below.

In [ ]:
import sys

print(sys.getsizeof("abc"))    # all-ASCII: 1 byte per character
print(sys.getsizeof("ábc"))    # one Latin-1 char: widens to 2 bytes per character
print(sys.getsizeof("🙂bc"))   # one emoji: widens to 4 bytes per character

## 4. Reading Text from Files

Corpora usually live in files. We will create one small article on disk first, then read it back — always naming the encoding explicitly, because the default varies by operating system.

In [ ]:
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)

article = """Tea Gardens of Sylhet
Sylhet sits in the north-east of Bangladesh, wrapped in rolling tea gardens.
Workers pick leaves at dawn, when the air is cool and the light is soft.
A single garden can employ hundreds of families across generations.
Tourists arrive in winter, ride open jeeps between rows of bright green bushes,
and drink cups of strong milk tea while fog lifts off the hills."""

path = Path("sample_data") / "article.txt"
path.write_text(article, encoding="utf-8")
print(f"wrote {len(article)} characters to {path}")

In [ ]:
from pathlib import Path

path = Path("sample_data") / "article.txt"

text = path.read_text(encoding="utf-8")            # whole file as one string
lines = text.splitlines()                          # list of lines, no "\n" left

print(lines[0])                                    # the title line
print("characters:", len(text), "| lines:", len(lines))
print("starts with title:", text.startswith("Tea Gardens"))

> ⚠️ Always pass `encoding="utf-8"`. On Windows the implicit default is often `cp1252`, so files containing `é` or 🙂 raise `UnicodeDecodeError` — or worse, decode silently into mojibake like `Ã©`. Name the encoding every single time.

## 5. Building a Corpus

A **corpus** is simply a collection of documents — a list of strings is enough to start. Below is our running dataset for the whole module: twelve authored product reviews and support messages, deliberately mixing praise, complaints, questions, and different writing styles.

In [ ]:
CORPUS = [
    "The battery life is amazing, easily two days of heavy use.",
    "Screen scratches way too easily. Very disappointed.",
    "Delivery was fast but the box arrived crushed.",
    "Camera quality at night is poor, lots of noise.",
    "Support fixed my billing issue in ten minutes. Impressive!",
    "Why did the price drop right after I bought it? So annoying.",
    "Sound quality is surprisingly rich for such a small speaker.",
    "My order has not arrived after two weeks. Where is it?",
    "Setup took five minutes and the app is genuinely easy to use.",
    "The strap snapped on day three. Cheap materials.",
    "Customer service kept me on hold for an hour.",
    "Great value for money, I bought a second one as a gift.",
]

print(len(CORPUS), "documents")
for i, doc in enumerate(CORPUS[:3], start=1):
    print(i, "->", doc)

## 6. Corpus Statistics

Before modeling anything, measure the material: how many words exist, how big the **vocabulary** (set of unique words) is, and which words dominate. The ratio of unique words to total words is called the **type-token ratio** — a quick measure of lexical richness.

**Syntax:**

```python
from collections import Counter

counts = Counter(iterable_of_words)
counts.most_common(20)     # [(word, n), ...] biggest first
```

In [ ]:
from collections import Counter

CORPUS = [
    "The battery life is amazing, easily two days of heavy use.",
    "Screen scratches way too easily. Very disappointed.",
    "Delivery was fast but the box arrived crushed.",
    "Camera quality at night is poor, lots of noise.",
    "Support fixed my billing issue in ten minutes. Impressive!",
    "Why did the price drop right after I bought it? So annoying.",
    "Sound quality is surprisingly rich for such a small speaker.",
    "My order has not arrived after two weeks. Where is it?",
    "Setup took five minutes and the app is genuinely easy to use.",
    "The strap snapped on day three. Cheap materials.",
    "Customer service kept me on hold for an hour.",
    "Great value for money, I bought a second one as a gift.",
]

flat = [w.strip(".,!?") for doc in CORPUS for w in doc.lower().split()]

total_words = len(flat)
vocab = set(flat)
ttr = len(vocab) / total_words

print("documents:        ", len(CORPUS))
print("total words:      ", total_words)
print("vocabulary size:  ", len(vocab))
print("type-token ratio: ", round(ttr, 3), "<- richer text has a higher ratio")

In [ ]:
from collections import Counter

CORPUS = [
    "The battery life is amazing, easily two days of heavy use.",
    "Screen scratches way too easily. Very disappointed.",
    "Delivery was fast but the box arrived crushed.",
    "Camera quality at night is poor, lots of noise.",
    "Support fixed my billing issue in ten minutes. Impressive!",
    "Why did the price drop right after I bought it? So annoying.",
    "Sound quality is surprisingly rich for such a small speaker.",
    "My order has not arrived after two weeks. Where is it?",
    "Setup took five minutes and the app is genuinely easy to use.",
    "The strap snapped on day three. Cheap materials.",
    "Customer service kept me on hold for an hour.",
    "Great value for money, I bought a second one as a gift.",
]
flat = [w.strip(".,!?") for doc in CORPUS for w in doc.lower().split()]
counts = Counter(flat)

for word, n in counts.most_common(10):
    print(f"{word:>12} | {n} {'#' * n}")

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

CORPUS = [
    "The battery life is amazing, easily two days of heavy use.",
    "Screen scratches way too easily. Very disappointed.",
    "Delivery was fast but the box arrived crushed.",
    "Camera quality at night is poor, lots of noise.",
    "Support fixed my billing issue in ten minutes. Impressive!",
    "Why did the price drop right after I bought it? So annoying.",
    "Sound quality is surprisingly rich for such a small speaker.",
    "My order has not arrived after two weeks. Where is it?",
    "Setup took five minutes and the app is genuinely easy to use.",
    "The strap snapped on day three. Cheap materials.",
    "Customer service kept me on hold for an hour.",
    "Great value for money, I bought a second one as a gift.",
]
flat = [w.strip(".,!?") for doc in CORPUS for w in doc.lower().split()]
top20 = Counter(flat).most_common(20)[::-1]        # reversed so biggest ends on top

words = [w for w, _ in top20]
freqs = [n for _, n in top20]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(words, freqs, color="#4477AA")
ax.set_title("Top 20 words in the corpus")
ax.set_xlabel("frequency")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

### 6.1 Zipf Teaser: Rank vs Frequency

Sort words by frequency and a law appears: the n-th most frequent word appears roughly `1/n` times as often as the first (Zipf's law). On a log-log plot this bends into a nearly straight declining line — a fingerprint of almost every natural-language collection ever measured.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

CORPUS = [
    "The battery life is amazing, easily two days of heavy use.",
    "Screen scratches way too easily. Very disappointed.",
    "Delivery was fast but the box arrived crushed.",
    "Camera quality at night is poor, lots of noise.",
    "Support fixed my billing issue in ten minutes. Impressive!",
    "Why did the price drop right after I bought it? So annoying.",
    "Sound quality is surprisingly rich for such a small speaker.",
    "My order has not arrived after two weeks. Where is it?",
    "Setup took five minutes and the app is genuinely easy to use.",
    "The strap snapped on day three. Cheap materials.",
    "Customer service kept me on hold for an hour.",
    "Great value for money, I bought a second one as a gift.",
]
flat = [w.strip(".,!?") for doc in CORPUS for w in doc.lower().split()]

freqs = np.array(sorted(Counter(flat).values(), reverse=True), dtype=float)
ranks = np.arange(1, len(freqs) + 1)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(ranks, freqs, "o", color="#4477AA")
ax.set_title("Zipf's law teaser: rank vs frequency")
ax.set_xlabel("rank (log scale)")
ax.set_ylabel("frequency (log scale)")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Sentence Splitting with Regex

Many tasks want sentences, not whole paragraphs. The tempting first attempt is "split at `.`, `!`, `?`" — and it fails immediately on abbreviations and decimals.

In [ ]:
import re

story = "Dr. Smith paid $3.50 for tea. Then he walked home!"

naive_parts = [p for p in re.split(r"[.!?]", story) if p]
for p in naive_parts:
    print(repr(p))
# Dr. lost its dot, $3.50 became $3 + 50, and the punctuation vanished

Three failures at once: the abbreviation `Dr.` was cut, the decimal `3.50` was destroyed, and the punctuation itself disappeared. We fix them one by one:

1. keep punctuation → split on the **whitespace after** punctuation using a lookbehind `(?<=[.!?])`;
2. decimals heal themselves, because a period inside `$3.50` has no space after it;
3. protect known abbreviations with extra negative lookbehinds like `(?<!Dr\.)`.

**Lookbehind** `(?<!...)` means: only match here if the text just before this position does NOT look like this. Each pattern must have fixed width — that is why we write one assertion per abbreviation.

**Example:**

In [ ]:
import re

story = "Dr. Smith paid $3.50 for tea. Then he walked home!"

sentence_re = re.compile(
    r"""
    (?<=[.!?])     # position sits right after . ! or ?
    (?<!Dr\.)      # ...but NOT right after these title abbreviations
    (?<!Mr\.)
    (?<!Mrs\.)
    (?<!Ms\.)
    (?<!St\.)
    \s+            # finally, split on the whitespace that follows
    """,
    re.VERBOSE,
)

for s in sentence_re.split(story):
    print(repr(s))

In [ ]:
import re

sentence_re = re.compile(
    r"""
    (?<=[.!?])
    (?<!Dr\.)(?<!Mr\.)(?<!Mrs\.)(?<!Ms\.)(?<!St\.)
    \s+
    """,
    re.VERBOSE,
)

harder = (
    "Mrs. Rahman ordered 2.5 kg of rice. It cost $12.75!! "
    "Was she happy? Yes. St. Martin tours were next."
)

for s in sentence_re.split(harder):
    print("-", s.strip())
# decimals survive, titles survive, real sentences still separate

Regex rules cover the common cases, but English always has another trick (*"etc.", ellipses…*, quotes). Production systems use trained sentence tokenizers instead:

```python
# pip install nltk
import nltk
nltk.download("punkt_tab")            # one-time model download

from nltk.tokenize import sent_tokenize
sent_tokenize("Dr. Smith paid $3.50 for tea. Then he walked home!")
# ['Dr. Smith paid $3.50 for tea.', 'Then he walked home!']
```

*(requires `pip install nltk` + the `punkt_tab` corpus download above)*

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Opening files without `encoding="utf-8"` | Windows defaults to `cp1252`; `é`/🙂 crash or turn into mojibake | always `open(..., encoding="utf-8")` |
| Trusting `len()` for display width | `len("🇧🇩")` is 2 (two indicators); 🙂 is 1 char but 4 bytes | decide what you mean: chars, bytes, or graphemes |
| Comparing user input without normalizing | `"café"` (NFC) `!=` `"café"` typed with a combining accent | `unicodedata.normalize("NFC", s)` at ingestion |
| Assuming `strip("!,.")` removes those everywhere | `strip` only trims from the two ENDS, character by character | use `replace()` or regex for interior cleanup |
| Building a vocabulary from raw `split()` | `"great!"` and `"great"` become different tokens | clean punctuation during tokenization (next lesson) |

```python
s = "...wow!!!"
print(s.strip("."))      # 'wow!!!'  -> strips leading dots only
print(s.replace(".", ""))  # 'wow!!!' with ALL dots gone
```

## 💡 Best Practices & Pro Tips

- Normalize Unicode (`NFC`) and fold case **once, at the boundary**, not scattered through your code.
- Always name encodings explicitly; treat `encoding="utf-8"` as part of the filename.
- Prefer `pathlib.Path` for file work — it composes with `/` and reads cleanly.
- Reach for `Counter` before writing a manual counting loop; it is faster and clearer.
- **AI-engineering relevance:** vocabulary size decides model size. Every tokenizer and embedding layer you will ever train has a row per unique word, so the profiling you did today (vocab size, TTR, Zipf curve) is literally capacity planning for future models. Garbage tokens in, garbage embeddings out.

## 📌 Summary

| Tool | What it does | Example |
|---|---|---|
| `s.lower()` / `s.strip()` | fold case / trim edge whitespace | `" Hi ".strip().lower()` → `"hi"` |
| `s.split()` / `sep.join(lst)` | words ←→ one string | `"a,b".split(",")` |
| `s.replace(a, b)` / `s.startswith(p)` | swap text / test prefix | `log.startswith("ERROR")` |
| `ord(c)` / `chr(n)` | character ←→ code point | `ord("A")` → `65` |
| `s.encode("utf-8")` | str → bytes (len differs!) | `len("🙂".encode())` → `4` |
| `unicodedata.normalize("NFC", s)` | one canonical spelling | fixes `é` vs `e\u0301` |
| `Counter(words).most_common(k)` | frequency ranking | top-k word list |
| type-token ratio | `len(set(words)) / len(words)` | lexical richness |
| `re.split` + lookbehind | sentence-ish splitting | `(?<=[.!?])(?<!Dr\.)\s+` |

**Key takeaways**

- Text must become clean, counted units before any modeling — that is 80% of applied NLP.
- Unicode has traps (bytes vs chars, flags, composed forms); normalize early and deliberately.
- Profile every corpus: totals, vocabulary, frequency curve tell you what the models will see.
- Naive regex gets you far; knowing precisely *why* it fails teaches you the fix.

## 🔗 Next Lesson

Raw text is now measurable — next we make it *clean and consistent*: [02_Text_Preprocessing](../02_Text_Preprocessing/notes.ipynb) builds the full cleaning-tokenizing-normalizing pipeline.